# ESIM, step 2: training the five embedding settings

ESIM (`model.py`): frozen word vectors → BiLSTM 200 → soft alignment both ways → `[a; ã; a−ã; a⊙ã]` → projection → BiLSTM 200 → mean + max pooling → MLP 200 (tanh, dropout 0.3) → one logit. Protocol (`training.py`, shared with the siamese BiLSTM): BCE, Adam 1e-3, at most 20 epochs, patience 3, 3 seeds.

| Setting | Embedding table |
|---|---|
| `mc2` | word2vec `min_count` 2 (the reproduction): words seen once share one learned `UNK` vector |
| `mc1` | word2vec `min_count` 1 |
| `ft_mc1` | fastText `min_count` 1 |
| `rand_mc2` | word2vec `min_count` 2 + one fixed random vector per dropped word (the kept setting) |
| `stack_mc1` | `[word2vec mc1; fastText mc1]`, 400-d |

Needs `01_word2vec.ipynb` first. About 3 min per epoch per seed on an M1 Pro (CPU). Writes `checkpoints/esim/<run>/` (git-ignored) and `predictions/<run>/` (history + val/test probabilities); runs already saved are skipped.

In [ ]:
import os
import sys
sys.path.append("..")   # repository root -> lets us import helpers/ and the model packages

import numpy as np
import pandas as pd

from helpers.pairs import load_pairs
from helpers.paths import PREDICTIONS
from esim.model import plain_table, random_rows_table, stacked_table, make_loaders, CHECKPOINTS
from esim.training import SEEDS, SETTINGS, run_training, save_run, health, rare_number_group
from siamese.training import broken_run_problems

DATASET = "overture_case2s"


## 1. Data

In [ ]:
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
splits = {"train": (train, train_label), "val": (val, val_label), "test": (test, test_label)}
all_addr = pd.concat([d[c] for d in (train, val, test) for c in ("address_a", "address_b")])   # the random rows' token list
print(len(train), len(val), len(test))

## 2. Train every missing run (5 settings × 3 seeds)

In [ ]:
tables = {   # setting -> builder of (vocab, matrix, config); built only if a seed is missing
    "mc2":       lambda: (*plain_table("word2vec", 2), {"w2v_kind": "word2vec", "w2v_min_count": 2}),
    "mc1":       lambda: (*plain_table("word2vec", 1), {"w2v_kind": "word2vec", "w2v_min_count": 1}),
    "ft_mc1":    lambda: (*plain_table("fasttext", 1), {"w2v_kind": "fasttext", "w2v_min_count": 1}),
    "rand_mc2":  lambda: (*random_rows_table(all_addr, min_count=2, seed=0)[:2], {"table": "word2vec_50_mc2 + random rows (seed 0)"}),
    "stack_mc1": lambda: (*stacked_table(min_count=1), {"table": "[word2vec_50_mc1; fasttext_50_mc1]"}),
}
for name, build in tables.items():
    run = SETTINGS[name]
    missing = [seed for seed in SEEDS if not os.path.exists(os.path.join(CHECKPOINTS, run, f"esim_seed_{seed}.pt"))]
    if not missing:
        continue
    vocab_s, matrix_s, config = build()
    loaders_s = make_loaders(vocab_s, splits)
    for seed in missing:
        print(f"{'=' * 20} {name} seed {seed} {'=' * 20}")
        history, best, best_state = run_training(matrix_s, loaders_s, val, val_label, seed)
        save_run(run, seed, history, best, best_state, vocab_s, matrix_s, loaders_s,
                 {**config, "emb_dim": matrix_s.shape[1], "dataset": DATASET})
        print(f"best: {best}\nproblems: {broken_run_problems(history)}")

## 3. Run health (epochs trained, broken-run checks; `[]` = healthy)

In [ ]:
health()

## 4. The rare-number group → `predictions/`
Pairs on the same street whose two house numbers both fall outside the word2vec `min_count` 2 vocabulary (paper, Tables 8–10). The group is fixed by that vocabulary, so it is saved once as a mask per split; `results/paper_tables.ipynb` reads the masks, so the tables need no word-vector files.

In [ ]:
for split, (df, _) in [("val", (val, val_label)), ("test", (test, test_label))]:
    grp = rare_number_group(df)
    np.save(os.path.join(PREDICTIONS, f"rare_number_group_{split}.npy"), grp)
    print(split, int(grp.sum()), "pairs in the group")